In [2]:
import pandas as pd

# Carregar os dados (ajuste o nome do arquivo se for .xlsx)
df = pd.read_csv("dados/online_retail_II.csv", encoding="ISO-8859-1")

# Visualizar as 5 primeiras linhas
print("Primeiras linhas:")
print(df.head())

# Verificar informações de colunas e tipos de dados
print("\nInformações do Dataset:")
print(df.info())

Primeiras linhas:
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  

Informações do Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       -------

In [3]:
# 1. Criar uma cópia para preservar os dados originais
df_limpo = df.copy()

# 2. Remover linhas sem Customer ID (essencial para RFM)
df_limpo = df_limpo.dropna(subset=['Customer ID'])

# 3. Converter Customer ID para número inteiro (remover o .0 do final)
df_limpo['Customer ID'] = df_limpo['Customer ID'].astype(int)

# 4. Converter InvoiceDate para o formato datetime do Pandas
df_limpo['InvoiceDate'] = pd.to_datetime(df_limpo['InvoiceDate'])

# 5. Filtrar apenas vendas válidas (Quantidade > 0 e Preço > 0)
df_limpo = df_limpo[(df_limpo['Quantity'] > 0) & (df_limpo['Price'] > 0)]

# 6. Criar a coluna de Valor Total da compra
df_limpo['ValorTotal'] = df_limpo['Quantity'] * df_limpo['Price']

# Visualizar o resultado do dataset limpo
print(f"Total de registros originais: {len(df)}")
print(f"Total de registros após a limpeza: {len(df_limpo)}")
print("\nPrimeiras linhas do dataset limpo:")
df_limpo.head()

Total de registros originais: 1067371
Total de registros após a limpeza: 805549

Primeiras linhas do dataset limpo:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,ValorTotal
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [4]:
# 1. Definir a data de referência (1 dia após a compra mais recente da base)
data_referencia = df_limpo['InvoiceDate'].max() + pd.Timedelta(days=1)

# 2. Agrupar por cliente e calcular Recência, Frequência e Valor Monetário
rfm = df_limpo.groupby('Customer ID').agg({
    'InvoiceDate': lambda date: (data_referencia - date.max()).days, # Recência
    'Invoice': 'nunique',                                           # Frequência
    'ValorTotal': 'sum'                                             # Monetário
}).reset_index()

# 3. Renomear as colunas
rfm.columns = ['Customer ID', 'Recencia', 'Frequencia', 'Monetario']

# Visualizar os 5 primeiros clientes processados
print(f"Total de clientes únicos analisados: {len(rfm)}")
print("\nPrimeiros clientes com métricas RFM:")
rfm.head()

Total de clientes únicos analisados: 5878

Primeiros clientes com métricas RFM:


,Customer ID,Recencia,Frequencia,Monetario
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [5]:
# 1. Gerar as pontuações de 1 a 5 para R, F e M
# Para Recência: menores valores ganham notas maiores (labels=[5, 4, 3, 2, 1])
rfm['R_score'] = pd.qcut(rfm['Recencia'], q=5, labels=[5, 4, 3, 2, 1])

# Para Frequência e Monetário: maiores valores ganham notas maiores (labels=[1, 2, 3, 4, 5])
# Usamos rank(method='first') para desempatar clientes com a mesma frequência de compras
rfm['F_score'] = pd.qcut(rfm['Frequencia'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['Monetario'], q=5, labels=[1, 2, 3, 4, 5])

# 2. Criar uma coluna combinada R_score + F_score (ex: "55") para definir o segmento
rfm['RF_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str)

# 3. Criar a regra para nomear os Segmentos do Cliente (Mapeamento de Negócio)
segment_map = {
    r'[1-2][1-2]': 'Hibernando',              # Recência baixa e frequência baixa
    r'[1-2][3-4]': 'Em Risco',                # Comprava muito, mas sumiu
    r'[1-2]5': 'Não Podemos Perdê-los',      # Eram os maiores compradores, mas faz tempo que não voltam
    r'3[1-2]': 'Prestes a Dormir',           # Recência mediana e compraram poucas vezes
    r'33': 'Precisa de Atenção',             # Clientes médios em tudo
    r'[3-4][4-5]': 'Leais',                   # Compram com boa frequência e voltaram recente
    r'41': 'Promissores',                    # Compraram recentemente pela primeira vez
    r'51': 'Novos Clientes',                 # Acabaram de fazer a primeira compra
    r'[4-5][2-3]': 'Potencialmente Leais',    # Clientes recentes com potencial de crescimento
    r'5[4-5]': 'Campeões'                     # Compram direto e acabaram de comprar
}

rfm['Segmento'] = rfm['RF_Score'].replace(segment_map, regex=True)

# Visualizar a distribuição dos segmentos
print("Quantidade de clientes por segmento:")
print(rfm['Segmento'].value_counts())

# Mostrar os primeiros clientes categorizados
rfm.head()

Quantidade de clientes por segmento:
Segmento
Hibernando               1523
Leais                    1161
Campeões                  837
Em Risco                  753
Potencialmente Leais      714
Prestes a Dormir          385
Precisa de Atenção        266
Promissores               114
Não Podemos Perdê-los      71
Novos Clientes             54
Name: count, dtype: int64


,Customer ID,Recencia,Frequencia,Monetario,R_score,F_score,M_score,RF_Score,Segmento
0,12346,326,12,77556.46,2,5,5,25,Não Podemos Perdê-los
1,12347,2,8,5633.32,5,4,5,54,Campeões
2,12348,75,5,2019.40,3,4,4,34,Leais
3,12349,19,4,4428.69,5,3,5,53,Potencialmente Leais
4,12350,310,1,334.40,2,1,2,21,Hibernando


In [6]:
# 1. Salvar a tabela fato (transações limpas)
df_limpo.to_csv("dados/fato_vendas.csv", index=False)

# 2. Salvar a tabela dimensão de clientes (RFM)
rfm.to_csv("dados/dim_clientes_rfm.csv", index=False)

print("Arquivos exportados com sucesso para a pasta 'dados'!")

Arquivos exportados com sucesso para a pasta 'dados'!
